In [30]:
# ...existing code...
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
# removed: from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')
# ...existing code...

In [31]:
df= pd.read_csv('/Users/vickeykarthik/Desktop/mlproject/notebook/data/Data/Transaction.csv')

In [5]:
df.head()

,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0


In [32]:
X = df.drop(columns=['is_fraud'],axis=1)

In [33]:
y = df['is_fraud']

In [34]:
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
         ("StandardScaler", numeric_transformer, num_features),        
    ]
)

In [35]:
X = preprocessor.fit_transform(X)

In [36]:
X.shape

(10000, 21)

In [37]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((8000, 21), (2000, 21))

In [38]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [39]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "CatBoosting Regressor": CatBoostRegressor(verbose=False),
    "AdaBoost Regressor": AdaBoostRegressor()
}
model_list = []
r2_list =[]

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    r2_list.append(model_test_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 0.0738
- Mean Absolute Error: 0.0573
- R2 Score: 0.8817
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.0760
- Mean Absolute Error: 0.0580
- R2 Score: 0.8918


Lasso
Model performance for Training set
- Root Mean Squared Error: 0.2146
- Mean Absolute Error: 0.0921
- R2 Score: 0.0000
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.2310
- Mean Absolute Error: 0.0994
- R2 Score: -0.0012


Ridge
Model performance for Training set
- Root Mean Squared Error: 0.0738
- Mean Absolute Error: 0.0573
- R2 Score: 0.8817
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 0.0759
- Mean Absolute Error: 0.0580
- R2 Score: 0.8918


K-Neighbors Regressor
Model performance for Training set
- Root Mean Squared Error: 0.0000
- Mean Absolute Error: 0.0000
- R2 Score: 1.0000
----------------------

In [40]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)

,Model Name,R2_Score
3,K-Neighbors Regressor,1.000000
4,Random Forest Regressor,1.000000
6,AdaBoost Regressor,1.000000
5,CatBoosting Regressor,0.999648
2,Ridge,0.891798
0,Linear Regression,0.891753
1,Lasso,-0.001238


In [42]:
pred_df = pd.DataFrame({'Actual Value': y_test, 'Predicted Value': y_test_pred, 'Difference': y_test - y_test_pred})
pred_df

,Actual Value,Predicted Value,Difference
6252,0,0.0,0.0
4684,0,0.0,0.0
1731,1,1.0,0.0
4742,0,0.0,0.0
4521,0,0.0,0.0
...,...,...,...
6412,0,0.0,0.0
8285,1,1.0,0.0
7853,0,0.0,0.0
1095,0,0.0,0.0
